In [1]:
import numpy as np
import cv2
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
import math

def load_images(binary_path, embedding_path):
    binary_data = np.load(binary_path)
    embedding_data = np.load(embedding_path)

    binary_image = binary_data['binary_image']
    embedding_image_original = embedding_data['embedding_image']

    embedding_image = embedding_image_original[:, :, 0:3]

    return binary_image, embedding_image

def create_masked_embedding_image(binary_image, embedding_image):
    masked_embedding_image = np.zeros_like(embedding_image)
    mask = binary_image > 0
    masked_embedding_image[mask] = embedding_image[mask]

    return masked_embedding_image

def extract_features(masked_embedding_image):
    h, w, _ = masked_embedding_image.shape
    mask = np.any(masked_embedding_image != [0, 0, 0], axis=-1)

    pixels = masked_embedding_image[mask].reshape(-1, 3)
    coords = np.column_stack(np.nonzero(mask))

    features = np.concatenate([pixels, coords], axis=1)

    return features, coords

def perform_dbscan_clustering(features, coords, eps=30, min_samples=100):
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(features)
    labels = db.labels_

    return labels

def create_clustered_image(masked_embedding_image, labels, coords):
    clustered_img = np.zeros_like(masked_embedding_image)
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
    colors = np.random.randint(0, 255, size=(n_clusters, 3))

    for label, (x, y) in zip(labels, coords):
        if label != -1:
            clustered_img[x, y] = colors[label % n_clusters]

    return clustered_img, unique_labels, labels

def process_clusters(masked_embedding_image, labels, coords, unique_labels):
    cluster_imgs = []
    cluster_means = []
    valid_labels = []

    for cluster_idx in unique_labels:
        if cluster_idx == -1:
            continue
        cluster_img = np.zeros_like(masked_embedding_image)
        cluster_coords = []

        for label, (x, y) in zip(labels, coords):
            if label == cluster_idx:
                cluster_img[x, y] = 255
                cluster_coords.append((x, y))

        cluster_imgs.append(cluster_img)
        cluster_coords = np.array(cluster_coords)
        cluster_mean = np.mean(cluster_coords, axis=0)
        cluster_means.append(cluster_mean)
        valid_labels.append(cluster_idx)

    return cluster_imgs, cluster_means, valid_labels

def process_image(image):
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, binary_image = cv2.threshold(gray_image, 1, 255, cv2.THRESH_BINARY)
    edges = cv2.Canny(binary_image, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=30, minLineLength=40, maxLineGap=10)
    blank_image = np.zeros_like(image)

    if lines is not None and len(lines) > 0:
        x1, y1, x2, y2 = lines[0][0]
        extension_length = 500
        angle = np.arctan2(y2 - y1, x2 - x1)
        x1_ext = int(x1 - extension_length * np.cos(angle))
        y1_ext = int(y1 - extension_length * np.sin(angle))
        x2_ext = int(x2 + extension_length * np.cos(angle))
        y2_ext = int(y2 + extension_length * np.sin(angle))
        cv2.line(blank_image, (x1_ext, y1_ext), (x2_ext, y2_ext), (255, 255, 255), 2)

    return blank_image, lines[0][0] if lines is not None and len(lines) > 0 else None

def max_count(image):
    h, w = image.shape[0], image.shape[1]
    max_y = 0
    max_count = 0
    for i in range(h):
        count = 0
        for j in range(w-1):
            if image[h-1-i][j][0] == 0 and image[h-1-i][j+1][0] != 0:
                count += 1
        if max_count < count:
            max_count = count
            max_y = h-1-i
    return max_y

def find_intersection_y(line_image, green_y):
    h, w = line_image.shape[:2]
    line_points = []
    for x in range(w):
        while line_image[green_y, x][0] != 0:
            line_points.append(x) 
    
    if line_points != []:
        return int(np.mean(line_points))
    return None

def generate_intersection_dict(cluster_imgs, point):
    accumulated_image = np.zeros_like(cluster_imgs[0])
    for i in range(len(cluster_imgs)):
        processed_image, _ = process_image(cluster_imgs[i])
        accumulated_image = cv2.add(accumulated_image, processed_image)
    max_y = max_count(accumulated_image)
    intersection_dict = {}

    for i in range(len(cluster_imgs)):
        processed_image, _ = process_image(cluster_imgs[i])
        intersection_x = find_intersection_y(processed_image, point[1])
        intersection_dict[f'Cluster_{i}'] = intersection_x

    return intersection_dict, accumulated_image, max_y

def find_intersecting_lines(point, intersection_dict):
    x, y = point
    intersection_dict = {key: value for key, value in intersection_dict.items() if value is not None}
    sorted_intersections = sorted(intersection_dict.items(), key=lambda item: item[1])

    for i in range(len(sorted_intersections) - 1):
        line1, x1 = sorted_intersections[i]
        line2, x2 = sorted_intersections[i + 1]
        if x1 <= x <= x2:
            return line1, line2, x1, x2
    return None, None, None, None

def calculate_percentage_closeness(point, x1, x2):
    x, y = point
    total_distance = abs(x2 - x1)
    distance_to_x1 = abs(x - x1)
    distance_to_x2 = abs(x - x2)
    percentage_closeness_x1 = (distance_to_x2 / total_distance) * 100
    percentage_closeness_x2 = (distance_to_x1 / total_distance) * 100
    return percentage_closeness_x1, percentage_closeness_x2




In [2]:
tracked_data = np.load('content/tracker/tracked_data.npy', allow_pickle=True)
def return_coordinate(framenumber):
    converted_data = {}
    for key, value in tracked_data[framenumber].items():
        # 첫 번째 값을 제외한 나머지 값을 취하고 int로 변환
        converted_data[key] = value[:].astype(int)
    converted_data2 = {}
    for key, value in converted_data.items():
        x_value = int(value[3])
        converted_values = [((int((value[0]*512/1920+value[2]*512/1920)/2)), int(x_value*256/1080))]
        converted_data2[key] = converted_values

    return converted_data2

In [3]:
def function(binary_path, embedding_path, img_path, point):
    return_result=[(point[1], point[0])]
    binary_image, embedding_image = load_images(binary_path, embedding_path)
    masked_embedding_image = create_masked_embedding_image(binary_image, embedding_image)
    features, coords = extract_features(masked_embedding_image)
    labels = perform_dbscan_clustering(features, coords)
    clustered_img, unique_labels, labels = create_clustered_image(masked_embedding_image, labels, coords)
    cluster_imgs, cluster_means, valid_labels = process_clusters(masked_embedding_image, labels, coords, unique_labels)

    intersection_dict, accumulated_image, max_y = generate_intersection_dict(cluster_imgs, point)
    # Sort the values and create rankings
    intersection_dict={k: v for k, v in intersection_dict.items() if v is not None}
    sorted_values = sorted(intersection_dict.values())

    original_img = cv2.imread(img_path)
    resized_img = cv2.resize(original_img, (512, 256))
    resized_img_rgb = cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB)
    plt.imshow(resized_img_rgb)
    mask = cv2.inRange(accumulated_image, np.array([255, 255, 255]), np.array([255, 255, 255]))
    masked_image = cv2.bitwise_and(accumulated_image, accumulated_image, mask=mask)
    plt.imshow(masked_image, cmap='gray', alpha=0.5)
    # plt.imshow(cv2.cvtColor(accumulated_image, cv2.COLOR_BGR2RGB))
    plt.title('Accumulated Extended Lines')
    for temp in sorted_values:
        return_result.append((point[1], temp))
        plt.plot(temp, point[1], 'ro', label='Point', markersize=2)
    # Create a dictionary to map original values to their ranks

    cv2.line(accumulated_image, (0, max_y), (accumulated_image.shape[1], max_y), (0, 255, 0), 2)

    plt.plot(point[0], point[1], 'bo', label='Point', markersize=2)
    plt.axis('off')
    plt.show()
    return return_result

In [4]:
result={}
i=20
print(i)
temp_dict=return_coordinate(i)
print(temp_dict)
binary_path = f'content/lanenet/{i:06d}binary_image.npz'
embedding_path = f'content/lanenet/{i:06d}embedding_image.npz'
img_path = f'content/tracker/{i:06d}.png'
for key, value in temp_dict.items():
    value_0=value[0]
    try:
        temp=function(binary_path, embedding_path, img_path, value_0)
        temp=[i]+temp
        if key not in result:
            result[key]=[]
            result[key].append(temp)
            print(temp)
        else :
            result[key].append(temp)
            print(temp)
        print(key)
    except Exception as e:
        print(e)

20
{'vehicle0': [(288, 119)], 'vehicle1': [(255, 124)], 'vehicle8': [(250, 112)]}


KeyboardInterrupt: 

In [ ]:
from collections import Counter

def most_common_length(nested_list):
    # 각 내부 리스트의 길이를 계산합니다.
    lengths = [len(inner_list) for inner_list in nested_list]

    # 길이들의 빈도를 계산합니다.
    length_counts = Counter(lengths)

    # 최빈값을 찾습니다.
    most_common_length = length_counts.most_common(1)[0][0]

    return most_common_length

for key, value in result.items():
  common_length=most_common_length(value)
  for i in value:
    if len(i)!=common_length:
      value.remove(i)

In [ ]:
result['vehicle0']

[[0, (118, 280), (118, 232), (118, 271), (118, 274)],
 [1, (118, 280), (118, 232), (118, 254), (118, 273)],
 [2, (119, 280), (119, 246), (119, 258), (119, 274)],
 [3, (119, 281), (119, 242), (119, 254), (119, 298)],
 [5, (119, 281), (119, 241), (119, 273), (119, 276)],
 [6, (119, 282), (119, 246), (119, 284), (119, 299)],
 [7, (119, 282), (119, 242), (119, 254), (119, 300)],
 [8, (119, 283), (119, 231), (119, 255), (119, 271)],
 [9, (119, 283), (119, 231), (119, 255), (119, 276)],
 [10, (119, 283), (119, 246), (119, 254), (119, 271)],
 [11, (119, 284), (119, 247), (119, 271), (119, 299)],
 [12, (119, 284), (119, 246), (119, 256), (119, 301)],
 [13, (119, 285), (119, 246), (119, 271), (119, 302)],
 [14, (119, 285), (119, 247), (119, 270), (119, 299)],
 [15, (119, 285), (119, 231), (119, 272), (119, 304)],
 [17, (119, 286), (119, 247), (119, 256), (119, 272)],
 [18, (119, 287), (119, 244), (119, 271), (119, 302)],
 [19, (119, 287), (119, 242), (119, 271), (119, 276)],
 [20, (119, 288), (

In [ ]:
import json

# JSON 파일로 저장
with open('result.json', 'w') as json_file:
    json.dump(result, json_file)


In [ ]:
!